In [1]:
import numpy as np
import pandas as pd
import spacy
import codecs, sys
import random
from collections import Counter
import pickle
from nltk.tokenize import word_tokenize


In [ ]:
import pandas as pd
import spacy
from transformers import BertTokenizer
from nltk.tokenize import word_tokenize
import nltk
from collections import Counter
import pickle

# Ensure necessary NLTK data is downloaded
nltk.download('punkt')

class Tokenizer:
    def __init__(self, dictionary_path=None):
        """
        Initialize the Tokenizer class with optional dictionary path.
        """
        self.nlp = spacy.blank('xx')  # Blank spaCy pipeline for multi-language support
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.dictionary = self.load_dictionary(dictionary_path) if dictionary_path else None

    def load_dictionary(self, filepath):
        """
        Load a dictionary from a pickle file.
        """
        try:
            with open(filepath, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Error loading dictionary: {e}")
            return None

    def tokenize(self, text, model_name='spacy'):
        """
        Tokenize the input text using the specified model.
        Supported models: 'bert', 'nltk', 'spacy', 'dictionary'.
        """
        if model_name == 'bert':
            return self._tokenize_bert(text)
        elif model_name == 'nltk':
            return self._tokenize_nltk(text)
        elif model_name == 'spacy':
            return self._tokenize_spacy(text)
        elif model_name == 'dictionary':
            return self._tokenize_dictionary(text)
        else:
            raise ValueError(f"Unsupported model: {model_name}")

    def _tokenize_bert(self, text):
        """
        Tokenize using BERT tokenizer.
        """
        return self.bert_tokenizer.tokenize(text)

    def _tokenize_nltk(self, text):
        """
        Tokenize using NLTK's word_tokenize.
        """
        return word_tokenize(text)

    def _tokenize_spacy(self, text):
        """
        Tokenize using spaCy.
        """
        doc = self.nlp(text)
        return [token.text for token in doc]

    def _tokenize_dictionary(self, text):
        """
        Tokenize using a custom dictionary.
        """
        if not self.dictionary:
            raise ValueError("Dictionary not loaded. Please provide a valid dictionary path.")
        
        # Lowercase the text
        text = text.lower()

        # Remove punctuation
        punctuation = '''!()%\n٪-;۔،:\n\/'"\,“./؟_ء'''
        for char in punctuation:
            text = text.replace(char, '')

        # Split into words
        tokens = text.split()

        # Handle bigrams using the dictionary
        bi_tokens = []
        i = 0
        while i < len(tokens):
            if i + 1 < len(tokens) and f"{tokens[i]} {tokens[i + 1]}" in self.dictionary:
                bi_tokens.append(f"{tokens[i]} {tokens[i + 1]}")
                i += 2
            else:
                bi_tokens.append(tokens[i])
                i += 1

        return bi_tokens

    def tokenize_csv(self, csv_file, output_file, model_name='spacy'):
        """
        Read a CSV file, tokenize the 'paragraph' column, and save the result to a new CSV file.
        """
        # Read the CSV file
        df = pd.read_csv(csv_file)

        # Ensure the required columns exist
        if 'paragraph' not in df.columns or 'category' not in df.columns:
            raise ValueError("CSV must contain 'paragraph' and 'category' columns.")

        # Tokenize the 'paragraph' column
        df['tokenized_words'] = df['paragraph'].apply(lambda x: self.tokenize(x, model_name))

        # Save the updated DataFrame to a new CSV file
        df.to_csv(output_file, index=False)
        print(f"Tokenized data saved to {output_file}")

# Example usage
if __name__ == "__main__":
    # Path to the input CSV file
    input_csv = "input_data.csv"

    # Path to the output CSV file
    output_csv = "tokenized_data.csv"

    # Path to the dictionary file (optional)
    dictionary_path = "/content/dictionary.pkl"

    # Create an instance of Tokenizer
    tokenizer = Tokenizer(dictionary_path=dictionary_path)

    # Tokenize the CSV file using the desired model (e.g., 'bert', 'nltk', 'spacy', 'dictionary')
    tokenizer.tokenize_csv(input_csv, output_csv, model_name='spacy')